# OECD FDI Income: Chi-Square, Homogeneity, and Variance Checks

This notebook performs:

1. Chi-square independence and homogeneity tests
2. Variance homogeneity checks required before ANOVA-style mean comparisons
3. ANOVA or Welch ANOVA depending on assumption results
4. Post-hoc comparisons chosen according to the variance test outcomes


In [2]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from IPython.display import display, Markdown

ALPHA = 0.05


In [3]:
project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent

data_path = project_root / 'data' / 'processed' / 'OECD_FDI_Except_Pointless.csv'
df = pd.read_csv(data_path, sep=';', encoding='latin1', engine='python', on_bad_lines='skip')
df['OBS_VALUE_NUM'] = pd.to_numeric(
    df['OBS_VALUE'].astype(str).str.replace('.', '', regex=False),
    errors='coerce'
)
anova_df = df.dropna(subset=['OBS_VALUE_NUM']).copy()
display(Markdown(f'**Rows used in ANOVA-style comparisons:** {anova_df.shape[0]:,}'))


**Rows used in ANOVA-style comparisons:** 7,407

## Chi-Square Tests

The same contingency machinery can be interpreted either as a test of independence or as a test of homogeneity, depending on the substantive question.


In [4]:
def cramers_v(table):
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    r, k = table.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))


chi_specs = [
    ('TYPE_ENTITY', 'MEASURE_PRINCIPLE', 'independence'),
    ('TIME_PERIOD', 'TYPE_ENTITY', 'homogeneity'),
]

chi_rows = []
for row_var, col_var, interpretation_type in chi_specs:
    table = pd.crosstab(df[row_var].astype(str), df[col_var].astype(str))
    chi2, p_value, dof, expected = stats.chi2_contingency(table)
    chi_rows.append({
        'row_variable': row_var,
        'column_variable': col_var,
        'interpretation_type': interpretation_type,
        'chi2': chi2,
        'p_value': p_value,
        'dof': dof,
        'cramers_v': cramers_v(table),
    })

chi_df = pd.DataFrame(chi_rows)
chi_df.round(6)


,row_variable,column_variable,interpretation_type,chi2,p_value,dof,cramers_v
0,TYPE_ENTITY,MEASURE_PRINCIPLE,independence,0.0,1.0,2,0.0
1,TIME_PERIOD,TYPE_ENTITY,homogeneity,0.0,1.0,2,0.0


### Comment

Chi-square significance should always be interpreted together with Cramer's V. In large samples, very small practical deviations can still become statistically significant.


## Homogeneity of Variance Before Mean Comparisons

Levene's test and a second equal-variance check are used before choosing the ANOVA strategy.


In [5]:
def eta_squared(data, outcome, group):
    clean = data[[outcome, group]].dropna().copy()
    grand_mean = clean[outcome].mean()
    grouped = clean.groupby(group)[outcome]
    ss_between = ((grouped.mean() - grand_mean) ** 2 * grouped.size()).sum()
    ss_total = ((clean[outcome] - grand_mean) ** 2).sum()
    return ss_between / ss_total


def pairwise_welch(data, outcome, group):
    results = []
    groups = {k: v[outcome].dropna().to_numpy() for k, v in data.groupby(group)}
    pairs = list(combinations(groups.keys(), 2))
    raw_p = []
    temp = []
    for g1, g2 in pairs:
        stat, p_value = stats.ttest_ind(groups[g1], groups[g2], equal_var=False)
        temp.append((g1, g2, stat, p_value))
        raw_p.append(p_value)
    reject, p_adj, _, _ = multipletests(raw_p, alpha=ALPHA, method='holm')
    for (g1, g2, stat, p_value), decision, p_corr in zip(temp, reject, p_adj):
        results.append({
            'group_1': g1,
            'group_2': g2,
            'welch_t': stat,
            'raw_p': p_value,
            'holm_p': p_corr,
            'reject_h0': bool(decision),
        })
    return pd.DataFrame(results)


anova_specs = ['TYPE_ENTITY', 'MEASURE_PRINCIPLE']
variance_rows = []
anova_rows = []
posthoc_tables = {}

for group_col in anova_specs:
    grouped = [g['OBS_VALUE_NUM'].dropna().to_numpy() for _, g in anova_df.groupby(group_col)]
    lev = stats.levene(*grouped, center='median')
    bart = stats.bartlett(*grouped)
    variance_equal = (lev.pvalue >= ALPHA) and (bart.pvalue >= ALPHA)

    variance_rows.append({
        'group_variable': group_col,
        'levene_stat': lev.statistic,
        'levene_p': lev.pvalue,
        'bartlett_stat': bart.statistic,
        'bartlett_p': bart.pvalue,
        'equal_variance_supported': variance_equal,
    })

    if variance_equal:
        anova_res = anova_oneway(anova_df['OBS_VALUE_NUM'], groups=anova_df[group_col], use_var='equal')
        posthoc_tables[group_col] = pairwise_tukeyhsd(anova_df['OBS_VALUE_NUM'], anova_df[group_col]).summary()
        method_label = 'Classical ANOVA + Tukey HSD'
    else:
        anova_res = anova_oneway(anova_df['OBS_VALUE_NUM'], groups=anova_df[group_col], use_var='unequal')
        posthoc_tables[group_col] = pairwise_welch(anova_df, 'OBS_VALUE_NUM', group_col)
        method_label = 'Welch ANOVA + pairwise Welch-Holm'

    anova_rows.append({
        'group_variable': group_col,
        'selected_method': method_label,
        'F_statistic': anova_res.statistic,
        'p_value': anova_res.pvalue,
        'effect_size_eta_squared': eta_squared(anova_df, 'OBS_VALUE_NUM', group_col),
    })

variance_df = pd.DataFrame(variance_rows)
anova_results_df = pd.DataFrame(anova_rows)
display(variance_df.round(6))
display(anova_results_df.round(6))


,group_variable,levene_stat,levene_p,bartlett_stat,bartlett_p,equal_variance_supported
0,TYPE_ENTITY,885.159289,0.000000,1669.864251,0.000000,False
1,MEASURE_PRINCIPLE,0.613483,0.433504,0.134828,0.713478,True


,group_variable,selected_method,F_statistic,p_value,effect_size_eta_squared
0,TYPE_ENTITY,Welch ANOVA + pairwise Welch-Holm,712.952756,0.000000,0.108454
1,MEASURE_PRINCIPLE,Classical ANOVA + Tukey HSD,0.003179,0.955041,0.000000


## Post-Hoc Results

The post-hoc procedure is chosen according to the variance tests.


In [6]:
display(Markdown('### TYPE_ENTITY post-hoc'))
display(posthoc_tables['TYPE_ENTITY'])

display(Markdown('### MEASURE_PRINCIPLE post-hoc'))
display(posthoc_tables['MEASURE_PRINCIPLE'])


### TYPE_ENTITY post-hoc

,group_1,group_2,welch_t,raw_p,holm_p,reject_h0
0,ALL,ROU,-0.671507,5.019294e-01,5.019294e-01,False
1,ALL,RSP,29.370056,7.557343e-171,2.267203e-170,True
2,ROU,RSP,28.322322,2.771503e-157,5.543005e-157,True


### MEASURE_PRINCIPLE post-hoc

group1,group2,meandiff,p-adj,lower,upper,reject
DI,DO,36589530361.334,0.955,-1235614566460.0305,1308793627182.6985,False


## Interpretation


In [7]:
comment = (
    '### Comment\n\n'
    'The decision path is explicit in this notebook: first check variance homogeneity, then choose the correct mean-comparison framework, and finally apply a compatible post-hoc procedure. '
    'If Levene and Bartlett support equal variances, classical ANOVA with Tukey HSD is appropriate. If they do not, Welch ANOVA with pairwise Welch-Holm comparisons is the safer choice. '
    'This prevents us from interpreting group mean differences under a variance structure that the data do not support.'
)
display(Markdown(comment))


### Comment

The decision path is explicit in this notebook: first check variance homogeneity, then choose the correct mean-comparison framework, and finally apply a compatible post-hoc procedure. If Levene and Bartlett support equal variances, classical ANOVA with Tukey HSD is appropriate. If they do not, Welch ANOVA with pairwise Welch-Holm comparisons is the safer choice. This prevents us from interpreting group mean differences under a variance structure that the data do not support.